# Notebook 06 — Weather Data Cleaning

This notebook cleans the raw monthly weather variables (Rainfall, Temperature, and Humidity), standardizing types, handling outliers, and outputting `weather_clean.csv`.

## 1. Import Packages & Local Configuration

In [1]:
import sys
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Ensure project root is in the path
sys.path.append(str(Path(os.getcwd()).parent))

import config
print("Processed Data Folder:", config.PROCESSED_DIR)

Processed Data Folder: D:\A1\coco\Coconut_Yield_Prediction\processed


## 2. Load Raw Weather Data

In [2]:
raw_weather_path = config.RAW_DIR / "weather_raw.csv"
if not raw_weather_path.exists():
    raise FileNotFoundError(f"{raw_weather_path.name} not found. Run Notebook 02 first.")

df = pd.read_csv(raw_weather_path)
print("Raw Weather Shape:", df.shape)
print(df.info())

Raw Weather Shape: (728, 11)
<class 'pandas.DataFrame'>
RangeIndex: 728 entries, 0 to 727
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   District            728 non-null    str    
 1   Year                728 non-null    int64  
 2   Month               728 non-null    int64  
 3   Rainfall            672 non-null    float64
 4   Temperature         728 non-null    float64
 5   Temperature_Max     728 non-null    float64
 6   Temperature_Min     728 non-null    float64
 7   Humidity            728 non-null    float64
 8   Solar_Radiation     728 non-null    float64
 9   Soil_Moisture_Top   728 non-null    float64
 10  Soil_Moisture_Root  728 non-null    float64
dtypes: float64(8), int64(2), str(1)
memory usage: 62.7 KB
None


## 3. Remove Duplicates & Check Types

In [3]:
# Drop exact duplicates
initial_len = len(df)
df = df.drop_duplicates(subset=["District", "Year", "Month"])
print(f"Removed {initial_len - len(df)} duplicate rows.")

# Verify types
df["Year"] = df["Year"].astype(int)
df["Month"] = df["Month"].astype(int)
for col in ["Rainfall", "Temperature", "Humidity"]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

Removed 0 duplicate rows.


## 4. Physical Boundaries & Outlier Detection

In [4]:
# Mask invalid values
df.loc[df["Rainfall"] < 0, "Rainfall"] = np.nan
df.loc[(df["Humidity"] < 0) | (df["Humidity"] > 100), "Humidity"] = np.nan
df.loc[(df["Temperature"] < 10) | (df["Temperature"] > 50), "Temperature"] = np.nan

# Outlier detection using Z-score per district and month
def replace_outliers_zscore(group, threshold=3.5):
    cleaned_group = group.copy()
    for col in ["Rainfall", "Temperature", "Humidity"]:
        mean = group[col].mean()
        std = group[col].std()
        if std > 0:
            z_scores = (group[col] - mean) / std
            outliers = np.abs(z_scores) > threshold
            cleaned_group.loc[outliers, col] = np.nan
    return cleaned_group

df_cleaned = df.copy()
for (dist, m), group in df.groupby(["District", "Month"]):
    cleaned_grp = replace_outliers_zscore(group)
    df_cleaned.loc[group.index, ["Rainfall", "Temperature", "Humidity"]] = cleaned_grp[["Rainfall", "Temperature", "Humidity"]]

print("Outliers replaced with NaN per column:\n", df.isna().sum() - df_cleaned.isna().sum())

Outliers replaced with NaN per column:
 District              0
Year                  0
Month                 0
Rainfall              0
Temperature           0
Temperature_Max       0
Temperature_Min       0
Humidity              0
Solar_Radiation       0
Soil_Moisture_Top     0
Soil_Moisture_Root    0
dtype: int64


## 5. Impute Missing Values

In [5]:
cols_to_impute = ["Rainfall", "Temperature", "Humidity"]
medians = df_cleaned.groupby(["District", "Month"])[cols_to_impute].transform("median")

for col in cols_to_impute:
    df_cleaned[col] = df_cleaned[col].fillna(medians[col])
    
print("Remaining missing values:", df_cleaned[cols_to_impute].isnull().sum().sum())

Remaining missing values: 56


## 6. Export Cleaned Weather Dataset

In [6]:
out_path = config.PROCESSED_DIR / "weather_clean.csv"
df_cleaned = df_cleaned[["District", "Year", "Month", "Rainfall", "Temperature", "Humidity"]]
df_cleaned.to_csv(out_path, index=False)
print(f"[SUCCESS] Cleaned weather dataset saved to {out_path.name}")

[SUCCESS] Cleaned weather dataset saved to weather_clean.csv
